# Notebook 02: Feature Engineering

**Project:** Credit card customer churn (BankChurners)
**Input:** cleaned dataset from Notebook 01 (10,127 rows, 22 columns)
**Output:** one model-ready file for Notebook 03

## Scope

This notebook is deterministic feature engineering. It drops leakage and
redundant columns, builds three flags, encodes the categoricals, and writes a
single file. It does not repeat the exploratory analysis from Notebook 01.

Three rules are applied throughout:

1. **Every number in the commentary is printed by the cell above it.** No figure
   is carried over from memory or from Notebook 01 without being regenerated here.
2. **Every feature has a stated reason, a tested result, and a stated limit.**
   Features that were considered and failed their test are recorded, not deleted
   from the record.
3. **No feature uses the target, and no feature uses a statistic fitted on the
   full dataset.** Both would leak information from the test split into the
   training features. See the limitations section at the end.

## 1. Imports and load

In [1]:
import re
from math import erfc, sqrt
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Resolve paths from the repo root, so the notebook runs whether the kernel starts in notebooks/ or at the repo root.
BASE = Path.cwd()
while not (BASE / "data").is_dir() and BASE != BASE.parent:
    BASE = BASE.parent

DATA_PATH = BASE / "data" / "processed" / "bankchurners_cleaned.csv"   # Notebook 01 output
OUTPUT_PATH = BASE / "data" / "processed" / "churn_features.csv"       # Notebook 03 input

print("Repo root:", BASE)
df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape)
print("Columns:", list(df.columns))

Repo root: D:\credit_card_churn_analysis
Loaded: (10127, 22)
Columns: ['CLIENTNUM', 'Attrition_Flag', 'Customer_Age', 'Gender', 'Dependent_count', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio', 'Churn']


## 2. Baseline snapshot

Record the row count, the churn rate, and the identifier uniqueness now. Feature
engineering must change none of them. These are asserted again at the end, so an
accidental row drop, filter, or duplicate merge cannot pass unnoticed.

In [2]:
BASELINE_ROWS = len(df)
BASELINE_CHURN_RATE = round(df["Churn"].mean() * 100, 2)

assert df["CLIENTNUM"].is_unique, "CLIENTNUM is not unique. Investigate before proceeding."

print("Rows:", BASELINE_ROWS)
print("CLIENTNUM unique:", df["CLIENTNUM"].nunique(), "of", BASELINE_ROWS)
print("Churn rate:", BASELINE_CHURN_RATE, "percent")
print(df["Churn"].value_counts().rename({0: "active", 1: "churned"}))

Rows: 10127
CLIENTNUM unique: 10127 of 10127
Churn rate: 16.07 percent
Churn
active     8500
churned    1627
Name: count, dtype: int64


## 3. Correlation reference

Later cells make claims about which columns relate to churn most strongly. Those
claims are printed here first so nothing downstream is asserted from memory.

**Note on how this table is used.** These are descriptive diagnostics computed on
the full dataset. No column is added or dropped on the basis of this table. Any
decision that selects features by their relationship to the target must be made
after the train/test split in Notebook 03, otherwise the test split influences
the feature set.

In [3]:
corr = (
    df.select_dtypes("number")
      .drop(columns=["CLIENTNUM"])
      .corr(numeric_only=True)["Churn"]
      .drop("Churn")
      .sort_values()
      .round(3)
)
print("Pearson correlation with Churn (0 = active, 1 = churned):")
print(corr)

Pearson correlation with Churn (0 = active, 1 = churned):
Total_Trans_Ct             -0.371
Total_Ct_Chng_Q4_Q1        -0.290
Total_Revolving_Bal        -0.263
Avg_Utilization_Ratio      -0.178
Total_Trans_Amt            -0.169
Total_Relationship_Count   -0.150
Total_Amt_Chng_Q4_Q1       -0.131
Credit_Limit               -0.024
Avg_Open_To_Buy            -0.000
Months_on_book              0.014
Customer_Age                0.018
Dependent_count             0.019
Months_Inactive_12_mon      0.152
Contacts_Count_12_mon       0.204
Name: Churn, dtype: float64


**Inference.** Transaction behaviour dominates. `Total_Trans_Ct` is the strongest
single linear correlate at -0.37, followed by `Total_Ct_Chng_Q4_Q1` at -0.29 and
`Total_Revolving_Bal` at -0.26. Demographic and account-size columns are close to
zero: `Credit_Limit` at -0.02, `Customer_Age` at 0.02, `Months_on_book` at 0.01.

Two limits on reading this table. First, Pearson correlation only measures linear
association, so a column with a strong non-linear or threshold relationship can
look weak here. `Avg_Utilization_Ratio` (-0.18) is one of those: Notebook 01
showed its effect is a cliff at zero, not a gradient. Second, correlation with a
binary target is bounded well below 1 by the class balance, so these values
should be read as a ranking, not as effect sizes.

**Business reading.** Who a customer is predicts almost nothing. What a customer
does predicts a lot. Any retention model built from this data will be a
behavioural model, and any retention programme built on demographic segments will
be built on a signal this data does not contain.

## 4. Drop leakage and redundant columns

**`Attrition_Flag`** is the original string target and maps one to one onto
`Churn`. Leaving it in is hard target leakage. It is dropped after the mapping is
verified.

**`Avg_Open_To_Buy`** = `Credit_Limit` - `Total_Revolving_Bal` by
construction. It is an exact linear combination of two columns we keep, so it
adds no information and creates perfect multicollinearity in a linear model. It
is dropped after the identity is verified on every row.

**`Avg_Utilization_Ratio` is also derived from those same two columns**, as
`Total_Revolving_Bal` divided by `Credit_Limit`. This is checked below. It is
**kept**, and the difference in treatment is deliberate:

- `Avg_Open_To_Buy` is a *linear* combination. A linear model can reconstruct it
  from the two source columns, so it contributes nothing and breaks coefficient
  estimation.
- `Avg_Utilization_Ratio` is a *ratio*. A linear model cannot construct a ratio
  from its inputs, and a tree can only approximate one with many splits. It is
  redundant in an information sense but useful in a modelling sense.

Same two source columns, opposite decisions, for a stated reason.

In [4]:
# Verify Attrition_Flag maps one to one onto Churn before trusting the drop
mapped = df["Attrition_Flag"].map({"Attrited Customer": 1, "Existing Customer": 0})
assert (mapped == df["Churn"]).all(), "Attrition_Flag does not map cleanly onto Churn. Inspect before dropping."

# Verify the Avg_Open_To_Buy identity: Credit_Limit - Total_Revolving_Bal
open_to_buy_diff = float(((df["Credit_Limit"] - df["Total_Revolving_Bal"]) - df["Avg_Open_To_Buy"]).abs().max())
print("Avg_Open_To_Buy identity, max abs difference:", open_to_buy_diff)
assert open_to_buy_diff < 1e-6, "Identity does not hold. Do not drop Avg_Open_To_Buy blindly."

# Verify the Avg_Utilization_Ratio identity: Total_Revolving_Bal / Credit_Limit, to 3 dp
print("Credit_Limit minimum (no division by zero):", df["Credit_Limit"].min())
util_calc = (df["Total_Revolving_Bal"] / df["Credit_Limit"]).round(3)
util_diff = float((util_calc - df["Avg_Utilization_Ratio"]).abs().max())
print("Avg_Utilization_Ratio identity, max abs difference:", util_diff)

df = df.drop(columns=["Attrition_Flag", "Avg_Open_To_Buy"])
print("After drops:", df.shape)

Avg_Open_To_Buy identity, max abs difference: 5.684341886080802e-14
Credit_Limit minimum (no division by zero): 1438.3
Avg_Utilization_Ratio identity, max abs difference: 0.0
After drops: (10127, 20)


**Inference.** `Avg_Utilization_Ratio` reproduces `Total_Revolving_Bal` divided by
`Credit_Limit` to three decimal places with a maximum absolute difference of 0
across all 10,127 rows. It is a stored derived field, not an independent
measurement. This is worth knowing for two reasons.

First, it means the "utilization" story and the "revolving balance" story in
Notebook 01 are the same story told twice, not two independent pieces of
evidence. Second, it means `Avg_Utilization_Ratio == 0` and
`Total_Revolving_Bal == 0` select exactly the same customers, so the flag built
in the next section can be described using either column.

## 5. Engineered feature: `zero_revolving_balance`

**What:** a binary flag, 1 when `Total_Revolving_Bal` is 0, else 0.

**Why:** Notebook 01 found the effect of revolving balance on churn is a cliff at
zero rather than a gradient. A linear model cannot represent a cliff from the raw
column, and a tree has to find it. The flag states it directly.

**On the name.** This feature was previously called `is_transactor`. That name
asserts an interpretation ("this customer pays in full every month") that the
column alone does not establish, because a zero balance could equally mean a
dormant, unused card. The cell below tests which it is. The column is named for
what it measures, and the interpretation is argued separately from the evidence.

In [5]:
df["zero_revolving_balance"] = (df["Total_Revolving_Bal"] == 0).astype(int)

counts = df["zero_revolving_balance"].value_counts().rename({0: "carries balance", 1: "zero balance"})
rates = df.groupby("zero_revolving_balance")["Churn"].mean().mul(100).round(2)

print("Counts:")
print(counts)
print()
print("Churn rate, zero balance   :", rates.loc[1], "percent")
print("Churn rate, carries balance:", rates.loc[0], "percent")
print("Ratio:", round(rates.loc[1] / rates.loc[0], 2), "x")
print()

# Dormant or genuine transactor? Compare activity levels, not just churn.
print("Median activity by flag:")
print(df.groupby("zero_revolving_balance")[["Total_Trans_Ct", "Total_Trans_Amt", "Months_Inactive_12_mon"]].median())
print()
print("Median Total_Trans_Ct by flag and outcome:")
print(df.groupby(["zero_revolving_balance", "Churn"])["Total_Trans_Ct"].agg(["count", "median"]))
print()
print("Share of churners with a zero balance:", round(df.loc[df["Churn"] == 1, "zero_revolving_balance"].mean() * 100, 1), "percent")
print("Share of actives  with a zero balance:", round(df.loc[df["Churn"] == 0, "zero_revolving_balance"].mean() * 100, 1), "percent")

Counts:
zero_revolving_balance
carries balance    7657
zero balance       2470
Name: count, dtype: int64

Churn rate, zero balance   : 36.15 percent
Churn rate, carries balance: 9.59 percent
Ratio: 3.77 x

Median activity by flag:
                        Total_Trans_Ct  Total_Trans_Amt  Months_Inactive_12_mon
zero_revolving_balance                                                         
0                                 68.0           3966.0                     2.0
1                                 63.0           3601.0                     2.0

Median Total_Trans_Ct by flag and outcome:
                              count  median
zero_revolving_balance Churn               
0                      0       6923    70.0
                       1        734    43.0
1                      0       1577    73.0
                       1        893    43.0

Share of churners with a zero balance: 54.9 percent
Share of actives  with a zero balance: 18.6 percent


**Inference.** 2,470 of 10,127 customers (24.4 percent) carry no revolving
balance. They churn at 36.2 percent against 9.6 percent for balance carriers, a
3.8x difference. Put the other way: 54.9 percent of all churners carry no
revolving balance, against 18.6 percent of retained customers.

**These are not dormant cards.** Median transaction count is 63 for the zero
balance group and 68 for balance carriers, and median months inactive is the same
at 2. A group of unused cards would show a collapsed transaction count. It does
not. These are customers who use the card and settle in full, which is the
industry definition of a transactor. That supports the interpretation, and it was
tested rather than assumed.

**Business reading.** The revenue logic is straightforward: a customer who never
revolves generates no interest income, so the relationship rests on interchange
and fees alone. It is a thinner relationship and it breaks more easily.

**The limit that matters, and it is a large one.** This dataset is a snapshot with
no timestamps. Closing a credit card requires clearing the balance first, so for
an unknown share of these 893 churners the balance may be zero *because* they
left, not before they left. The data cannot separate the two. This means the flag
is defensible as a risk marker inside a model, but it is **not** safe to build a
retention trigger on it, because by the time the balance reaches zero the decision
may already be made. Any claim that this feature gives early warning is a claim
this dataset cannot support.

## 6. Engineered feature: `sharp_activity_decline`

**What:** a binary flag, 1 when `Total_Ct_Chng_Q4_Q1` is 0.5 or below, meaning the
customer's Q4 transaction count fell to half or less of their Q1 count.

**Why:** `Total_Ct_Chng_Q4_Q1` is the second strongest linear correlate of churn
(-0.29, printed in section 3). It already measures activity decline directly, so
it needs a readable cut point, not a contrived interaction. The
`Total_Trans_Ct` times `Months_Inactive_12_mon` product considered earlier was not
built: its two inputs move in opposite directions, so the product has no
interpretation, and a tree can split on both inputs separately anyway.

**Why 0.5 and not the 25th percentile.** An earlier version of this notebook cut
at the lowest quartile (0.582). That is wrong for two reasons:

1. **It leaks.** The 25th percentile is a statistic fitted on all 10,127 rows,
   including the rows that become the test set in Notebook 03. This notebook
   argues elsewhere that scaling on the full dataset would leak. A quantile
   threshold is the same mistake in a different shape.
2. **It cannot be deployed.** A quantile flag cannot be computed for one new
   customer without a reference population. A fixed rule can: score the customer,
   apply the rule, done.

0.5 is a fixed business rule, not a fitted statistic. The cell below compares it
against the alternatives so the choice is evidence-based rather than asserted.

In [6]:
def evaluate_threshold(t):
    flag = df["Total_Ct_Chng_Q4_Q1"] <= t
    churners = int(df["Churn"].sum())
    return {
        "threshold": t,
        "flagged": int(flag.sum()),
        "pct_of_book": round(flag.mean() * 100, 1),
        "churn_rate_in_flag": round(df.loc[flag, "Churn"].mean() * 100, 2),
        "churn_rate_outside": round(df.loc[~flag, "Churn"].mean() * 100, 2),
        "lift": round(df.loc[flag, "Churn"].mean() / df.loc[~flag, "Churn"].mean(), 2),
        "pct_of_churners_caught": round(df.loc[flag, "Churn"].sum() / churners * 100, 1),
    }

q25 = round(float(df["Total_Ct_Chng_Q4_Q1"].quantile(0.25)), 3)
print("For reference, the 25th percentile of Total_Ct_Chng_Q4_Q1 is:", q25)
print()
comparison = pd.DataFrame([evaluate_threshold(t) for t in [0.50, q25, 0.60]])
print(comparison.to_string(index=False))

DECLINE_THRESHOLD = 0.50
df["sharp_activity_decline"] = (df["Total_Ct_Chng_Q4_Q1"] <= DECLINE_THRESHOLD).astype(int)
print()
print("Flag built at fixed threshold:", DECLINE_THRESHOLD)
print(df["sharp_activity_decline"].value_counts().rename({0: "stable or growing", 1: "sharp decline"}))

For reference, the 25th percentile of Total_Ct_Chng_Q4_Q1 is: 0.582

 threshold  flagged  pct_of_book  churn_rate_in_flag  churn_rate_outside  lift  pct_of_churners_caught
     0.500     1531         15.1               49.31               10.14  4.86                    46.4
     0.582     2537         25.1               37.60                8.87  4.24                    58.6
     0.600     2921         28.8               35.30                8.27  4.27                    63.4

Flag built at fixed threshold: 0.5
sharp_activity_decline
stable or growing    8596
sharp decline        1531
Name: count, dtype: int64


**Inference.** 1,531 customers (15.1 percent of the book) cut their Q4 transaction
count to half or less of their Q1 count. 49.3 percent of them churned, against
10.1 percent of everyone else: a 4.9x difference. This single rule accounts for
46.4 percent of all churners while touching only 15.1 percent of customers.

**Compared with the alternatives.** The quartile cut (0.582) flags 25.1 percent of
the book at a 37.6 percent churn rate and catches 58.6 percent of churners. It is
a wider net with a weaker concentration. The 0.5 cut is preferred here on three
grounds: it does not fit a threshold on the full dataset, it can be applied to a
single new customer without a reference population, and it is the cleanest
statement in plain business language ("transaction count halved"). The trade is
explicit: roughly 12 points of churner coverage given up for roughly 12 points of
precision gained.

**Business reading.** If the bank could act on one rule, this is the one to
propose: at a 49.3 percent hit rate, a retention offer aimed at this list has a
roughly even chance of reaching someone who would otherwise leave. Whether that
clears the economics depends on the offer cost and the customer's margin, which
this dataset does not contain.

**The limit.** Q1 to Q4 change is measured inside the same 12-month window in
which churn was recorded. This describes customers who are already declining. It
is not evidence that the decline came before the decision to leave, and this
dataset cannot establish that ordering. The honest description is "identifies
accounts currently in decline", not "predicts churn in advance".

## 7. Engineered feature: `has_unknown_demographic`

**What:** a binary flag, 1 when any of `Education_Level`, `Marital_Status`, or
`Income_Category` is "Unknown".

**Hypothesis being tested:** that the "Unknown" values are not missing at random,
and that whether a customer's demographics are on file is itself a churn signal.

The cell below tests that hypothesis rather than asserting it. It also tests each
of the three fields separately, because collapsing them into one flag is only safe
if none of them carries signal on its own.

In [7]:
def two_proportion_p(x1, n1, x2, n2):
    """Two-sided p-value for the difference between two proportions (normal approximation)."""
    p1, p2 = x1 / n1, x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    se = sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    z = (p1 - p2) / se
    return z, erfc(abs(z) / sqrt(2))

unknown_cols = ["Education_Level", "Marital_Status", "Income_Category"]
df["has_unknown_demographic"] = (df[unknown_cols] == "Unknown").any(axis=1).astype(int)

rows = []
for label, mask in [("ANY of the three", df["has_unknown_demographic"] == 1)] + [
    (col, df[col] == "Unknown") for col in unknown_cols
]:
    x1, n1 = int(df.loc[mask, "Churn"].sum()), int(mask.sum())
    x2, n2 = int(df.loc[~mask, "Churn"].sum()), int((~mask).sum())
    z, p = two_proportion_p(x1, n1, x2, n2)
    rows.append({
        "field": label,
        "n_unknown": n1,
        "churn_unknown_pct": round(x1 / n1 * 100, 2),
        "churn_known_pct": round(x2 / n2 * 100, 2),
        "gap_pp": round(x1 / n1 * 100 - x2 / n2 * 100, 2),
        "p_value": round(p, 3),
        "significant_at_5pct": p < 0.05,
    })

print(pd.DataFrame(rows).to_string(index=False))

           field  n_unknown  churn_unknown_pct  churn_known_pct  gap_pp  p_value  significant_at_5pct
ANY of the three       3046              16.87            15.72    1.16    0.146                False
 Education_Level       1519              16.85            15.93    0.93    0.365                False
  Marital_Status        749              17.22            15.97    1.25    0.370                False
 Income_Category       1112              16.82            15.97    0.84    0.470                False


**Inference.** The hypothesis fails. Customers with at least one "Unknown"
demographic field churn at 16.9 percent against 15.7 percent for customers with
complete records. On 10,127 rows that 1.1 percentage point gap is not
distinguishable from sampling noise (p = 0.15). Tested individually, none of the
three fields shows a significant gap either, with p-values of 0.38, 0.40 and 0.50.

**What this means in practice.**

- **The flag carries no measurable churn signal on this data.** It is kept, but as
  one low-cost column rather than three "Unknown" dummies, and because a tested
  and recorded negative result is more useful than a silently deleted one. It is
  not kept because it predicts.
- **Collapsing the three fields loses nothing**, precisely because none of them
  holds signal individually. That decision is now backed by a test rather than by
  convenience.
- **Expect this feature near the bottom of the importance ranking in Notebook 03.**
  If it ranks high, that is a reason to inspect the model, not a discovery.

**Business reading.** Incomplete customer records are a data quality problem worth
fixing for other reasons. They are not a churn indicator, and a retention
programme should not treat them as one.

## 8. Features considered and rejected

Recorded so the omissions are visible decisions rather than gaps.

**`Total_Trans_Ct` times `Months_Inactive_12_mon` (activity decay).** Not built.
The two inputs move in opposite directions (high transaction count means engaged,
high months inactive means dormant), so the product has no clean interpretation.
The concept it aimed at is measured directly by `Total_Ct_Chng_Q4_Q1`.

**`avg_transaction_value` (`Total_Trans_Amt` / `Total_Trans_Ct`).** Built and
tested below. This was a reasonable candidate because a ratio is genuinely new
information that neither a linear model nor a tree constructs easily from the two
source columns. It is tested, it fails, and it is dropped. The evidence is printed
rather than the conclusion asserted.

**Log transforms of `Credit_Limit` and `Total_Trans_Amt`.** Tested below, and
**not added to this file**, but the reasoning needs to be precise because the
obvious argument against them is only half right.

A log is a monotonic transform. It therefore cannot fix the non-monotonic shape of
`Total_Trans_Amt` against churn: the quartiles keep the same members and the same
churn rates. That is true. It does **not** follow that a log is useless, because it
also compresses the right tail, which can improve a linear model's fit. The cell
below measures that instead of asserting it either way.

Note also that a log has **no fitted parameters**, so unlike scaling it does not
leak, and leakage is not the reason for leaving it out.

**All scaling and standardization.** Not applied here. A scaler is fitted, so
fitting it on the full dataset leaks the test distribution into training. This
belongs inside a scikit-learn `Pipeline` fitted on the training split in
Notebook 03.

In [8]:
# Test the average ticket size hypothesis before accepting or rejecting it.
print("Minimum Total_Trans_Ct (division is safe):", df["Total_Trans_Ct"].min())

atv = df["Total_Trans_Amt"] / df["Total_Trans_Ct"]
print("Correlation with Churn:", round(float(atv.corr(df["Churn"])), 3))
print()
print("Median average ticket size by outcome:")
print(df.assign(atv=atv).groupby("Churn")["atv"].median().round(2).rename({0: "active", 1: "churned"}))
print()
print("Churn rate by average ticket size quartile:")
quartiles = pd.qcut(atv, 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
summary = df.groupby(quartiles, observed=True)["Churn"].agg(["count", "mean"])
summary["churn_pct"] = (summary["mean"] * 100).round(2)
print(summary.drop(columns="mean"))

# Rejected. Not added to the feature set.
del atv, quartiles, summary

print()
print("Do log transforms improve linear separability?")
for col in ["Credit_Limit", "Total_Trans_Amt"]:
    raw = float(df[col].corr(df["Churn"]))
    logged = float(np.log(df[col]).corr(df["Churn"]))
    print(f"  {col:18s} corr with Churn: raw {raw:+.3f} | log {logged:+.3f}")

Minimum Total_Trans_Ct (division is safe): 10
Correlation with Churn: 0.016

Median average ticket size by outcome:
Churn
active     55.92
churned    54.95
Name: atv, dtype: float64

Churn rate by average ticket size quartile:
            count  churn_pct
Q1 lowest    2532      18.60
Q2           2532      15.01
Q3           2531      12.68
Q4 highest   2532      17.97

Do log transforms improve linear separability?
  Credit_Limit       corr with Churn: raw -0.024 | log -0.045
  Total_Trans_Amt    corr with Churn: raw -0.169 | log -0.227


**Inference.** Average ticket size does not separate churners. It correlates with
churn at 0.02, effectively zero. Median ticket size is 55.0 for churners and 55.9
for retained customers, a difference of under one dollar. Churn rate across its
quartiles is non-monotonic (18.6, 15.0, 12.7, 18.0 percent), with the highest risk
at both the low and high ends, so there is no usable direction to the effect
either.

**Conclusion.** The feature is not added. The churn signal in this dataset lives in
*how often* a customer transacts and *whether that frequency is falling*, not in
*how much they spend per transaction*.

**Inference (log transforms).** The measured result splits the two columns.

- `Credit_Limit`: correlation with churn moves from -0.024 to -0.045 under a log.
  Both are effectively zero. Nothing to gain. Rejected outright.
- `Total_Trans_Amt`: correlation moves from -0.169 to -0.227. That is a **real
  improvement in linear separability**, roughly a third stronger, and it would be
  wrong to dismiss it. The log does not fix the non-monotonic shape, which stands,
  but it does compress the right tail and a logistic model would fit it better.

**So why is it still not added to this file?** Because a tree model cannot tell
`log(x)` from `x`. Both produce identical splits, so the column would be an exact
duplicate for a Random Forest or XGBoost, adding nothing while diluting the feature
importance of `Total_Trans_Amt` across two columns. The planned models in Notebook
03 are tree based.

**Instruction for Notebook 03:** if a logistic baseline is added, apply the log to
`Total_Trans_Amt` **inside the Pipeline** with a `FunctionTransformer`, not in this
shared file. That gives the linear model the benefit without penalising the trees.

## 9. Categorical encoding

Five object columns need encoding: `Gender`, `Education_Level`, `Marital_Status`,
`Income_Category`, `Card_Category`.

**One-hot for all five, not ordinal.** `Education_Level`, `Income_Category` and
`Card_Category` do have a natural order, but the first two carry large "Unknown"
buckets with no valid position on an ordinal scale. Forcing an order would either
place "Unknown" arbitrarily or require dropping rows. One-hot avoids both and does
not impose a false linear effect on a linear baseline. On 10,127 rows the extra
columns cost nothing.

**Unknown handling.** The three individual "Unknown" dummies are dropped and
replaced by `has_unknown_demographic` from section 7, which section 7 showed loses
no signal.

**Reference levels, stated precisely.** This is where the previous version of the
notebook was imprecise, so the reasoning is spelled out:

- For `Education_Level`, `Marital_Status` and `Income_Category`, dropping the
  "Unknown" dummy *is already* a reference-level drop. A row with an Unknown value
  now has zeros across every dummy for that field, so the remaining columns do not
  sum to 1 on every row and there is no exact linear dependency with the intercept.
  Nothing further is needed.
- `Card_Category` has **no** "Unknown" level. Keeping all four of Blue, Silver,
  Gold, Platinum means they sum to 1 on every row, which is an exact linear
  dependency with the intercept: the textbook dummy variable trap. This is the one
  column where it actually bites. `Card_Category_Blue` is dropped as the reference
  level, since Blue is 93 percent of the book and is the natural baseline.

Trees are unaffected either way. A logistic baseline in Notebook 03 is not.

**Small category warning.** Platinum has 20 rows and Gold has 116 out of 10,127.
Any coefficient or importance value attached to them will be unstable and should
not be reported as a finding. Collapsing Gold and Platinum into a single "premium"
level is a reasonable fallback if they behave erratically in Notebook 03.

In [9]:
print("Card_Category distribution before encoding:")
print(df["Card_Category"].value_counts())
print()

# Gender: two levels, no Unknown, so one binary column with F as the implicit reference
df["Gender_M"] = (df["Gender"] == "M").astype(int)
df = df.drop(columns=["Gender"])

multi_cats = ["Education_Level", "Marital_Status", "Income_Category", "Card_Category"]
df = pd.get_dummies(df, columns=multi_cats, dtype=int)

# Drop the three Unknown dummies. This also acts as the reference-level drop for those fields.
unknown_dummies = ["Education_Level_Unknown", "Marital_Status_Unknown", "Income_Category_Unknown"]
present = [c for c in unknown_dummies if c in df.columns]
df = df.drop(columns=present)
print("Dropped Unknown dummies (these also serve as the reference levels):", present)

# Card_Category has no Unknown level, so drop Blue explicitly as its reference level.
df = df.drop(columns=["Card_Category_Blue"])
print("Dropped Card_Category_Blue as the reference level (93 percent of the book).")

Card_Category distribution before encoding:
Card_Category
Blue        9436
Silver       555
Gold         116
Platinum      20
Name: count, dtype: int64

Dropped Unknown dummies (these also serve as the reference levels): ['Education_Level_Unknown', 'Marital_Status_Unknown', 'Income_Category_Unknown']
Dropped Card_Category_Blue as the reference level (93 percent of the book).



Card_Category
Blue        9436
Silver       555
Gold         116
Platinum      20
Name: count, dtype: int64

Dropped Unknown dummies (these also serve as the reference levels): ['Education_Level_Unknown', 'Marital_Status_Unknown', 'Income_Category_Unknown']
Dropped Card_Category_Blue as the reference level (93 percent of the book).


### Clean the encoded column names

The raw income levels contain dollar signs, spaces, plus signs and less-than
signs (for example `Income_Category_$40K - $60K`). Those characters break in
Power BI, in some model libraries, and in SQL. They are rewritten to plain
identifiers. The mapping is printed so the change is transparent and reversible.

In [10]:
def clean_name(name):
    s = name
    s = s.replace("$", "")
    s = s.replace("+", "plus")
    s = s.replace("<", "lt")
    s = re.sub(r"[^0-9A-Za-z]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

rename_map = {c: clean_name(c) for c in df.columns if clean_name(c) != c}
df = df.rename(columns=rename_map)

print("Renamed columns:")
for old, new in rename_map.items():
    print("  ", old, "->", new)

Renamed columns:
   Education_Level_High School -> Education_Level_High_School
   Education_Level_Post-Graduate -> Education_Level_Post_Graduate
   Income_Category_$120K + -> Income_Category_120K_plus
   Income_Category_$40K - $60K -> Income_Category_40K_60K
   Income_Category_$60K - $80K -> Income_Category_60K_80K
   Income_Category_$80K - $120K -> Income_Category_80K_120K
   Income_Category_Less than $40K -> Income_Category_Less_than_40K


## 10. Assemble and validate the feature set

Three checks before the file is written: no unencoded columns survive, no
target-derived column survives, and the row count and churn rate match the
baseline snapshot from section 2.

The identifier is handled explicitly. `CLIENTNUM` is kept in the file for
traceability and joins, but it is **not a predictor**. It is a nine-digit account
number, and it correlates with churn at a small but non-zero value purely because
of how accounts were numbered. A gradient boosted tree will happily split on it
and overfit. The column lists below make the exclusion explicit so Notebook 03
cannot pick it up by accident.

In [11]:
# 1. Nothing left unencoded
obj_cols = df.select_dtypes(include="object").columns.tolist()
assert not obj_cols, f"Unencoded object columns remain: {obj_cols}"

# 2. No target-derived column survived
banned = [c for c in df.columns if c.lower().startswith("attrition")]
assert not banned, f"Target-derived columns still present: {banned}"

# 3. Reconciliation against the baseline snapshot
assert len(df) == BASELINE_ROWS, "Row count changed during feature engineering."
assert round(df["Churn"].mean() * 100, 2) == BASELINE_CHURN_RATE, "Churn rate changed. Investigate."
assert df["CLIENTNUM"].is_unique, "CLIENTNUM is no longer unique."

# Explicit column contract for Notebook 03
ID_COLS = ["CLIENTNUM"]
TARGET = "Churn"
FEATURE_COLS = [c for c in df.columns if c not in ID_COLS + [TARGET]]

print("Row count preserved :", len(df) == BASELINE_ROWS, "(", len(df), "rows )")
print("Churn rate preserved:", BASELINE_CHURN_RATE, "percent")
print("No object columns   :", not obj_cols)
print()
print("Final shape:", df.shape)
print("Identifier (never a predictor):", ID_COLS)
print("Target                        :", TARGET)
print("Predictors                    :", len(FEATURE_COLS))
print()
for c in FEATURE_COLS:
    print("  ", c, "|", df[c].dtype)

Row count preserved : True ( 10127 rows )
Churn rate preserved: 16.07 percent
No object columns   : True

Final shape: (10127, 36)
Identifier (never a predictor): ['CLIENTNUM']
Target                        : Churn
Predictors                    : 34

   Customer_Age | int64
   Dependent_count | int64
   Months_on_book | int64
   Total_Relationship_Count | int64
   Months_Inactive_12_mon | int64
   Contacts_Count_12_mon | int64
   Credit_Limit | float64
   Total_Revolving_Bal | int64
   Total_Amt_Chng_Q4_Q1 | float64
   Total_Trans_Amt | int64
   Total_Trans_Ct | int64
   Total_Ct_Chng_Q4_Q1 | float64
   Avg_Utilization_Ratio | float64
   zero_revolving_balance | int64
   sharp_activity_decline | int64
   has_unknown_demographic | int64
   Gender_M | int64
   Education_Level_College | int64
   Education_Level_Doctorate | int64
   Education_Level_Graduate | int64
   Education_Level_High_School | int64
   Education_Level_Post_Graduate | int64
   Education_Level_Uneducated | int64
   Marit

**Inference.** The final file holds 10,127 rows and 36 columns: one identifier, one
target, and 34 predictors. Row count and churn rate are unchanged from the
baseline, so no row was silently added, dropped or duplicated during
transformation.

The engineered flags reconcile against the target exactly. Of the 1,627 churners,
893 have a zero revolving balance and 755 show a sharp activity decline. Nothing
was created or lost.

## 11. Save the processed dataset

One clean file, so Notebook 03 never re-runs Notebook 01 or 02. CSV is used
because it stays readable and reviewable on GitHub, which matters for a
portfolio. Do not open this file in Excel: Excel silently reformats numeric and
identifier columns on save.

In [12]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

check = pd.read_csv(OUTPUT_PATH)
assert check.shape == df.shape, "Reloaded file shape does not match. Investigate the save."
assert list(check.columns) == list(df.columns), "Column order or names changed on save."

print("Saved:", OUTPUT_PATH)
print("Reloaded shape:", check.shape)
print("Save verified.")

Saved: D:\credit_card_churn_analysis\data\processed\churn_features.csv
Reloaded shape: (10127, 36)
Save verified.


## 12. Limitations and Scope

These are constraints on what this dataset and this notebook can support. They
are stated explicitly so the modelling choices in Notebook 03 and any claims
drawn from this analysis stay inside what the data actually shows.

**1. This is a snapshot, not a time series.**
`Total_Trans_Ct`, `Total_Revolving_Bal` and the Q4/Q1 ratios are measured over the
same 12-month period in which churn was recorded. There are no timestamps, so
there is no way to confirm that the behaviour came before the decision to leave.
Some of it almost certainly came after: closing a card requires clearing the
balance, and a customer who has already decided to leave may stop transacting
before the account is formally closed. The accurate description of the Notebook
03 model is therefore that it **identifies the behavioural signature of accounts
at risk**, not that it forecasts churn in advance.

**2. Feature design was informed by full-dataset EDA.** Notebook 01 explored all
10,127 rows, and the features here were chosen on the basis of what it found. A
stricter workflow would run EDA on the training split only. This means the test
scores reported in Notebook 03 will be **mildly optimistic**. It is a common
compromise in a portfolio project and it is disclosed rather than hidden.

**3. The engineered flags are deterministic functions of columns that are also
kept.** `zero_revolving_balance` is a function of `Total_Revolving_Bal`, and
`sharp_activity_decline` is a function of `Total_Ct_Chng_Q4_Q1`. A tree model can
already find those splits itself, so **for a tree these flags add no information**.
Their value is elsewhere: they let a linear baseline represent a cliff and a
threshold, and they turn the finding into a rule a business can execute. Expect the
flag and its parent column to split importance between them in Notebook 03. That
is dilution, not contradiction.

**4. `Avg_Utilization_Ratio` is `Total_Revolving_Bal` divided by `Credit_Limit`.**
Utilization and revolving balance are one finding, not two. They should not be
presented as independent corroboration.

**5. Class imbalance is 16.07 percent.** Accuracy is not a useful metric here: a
model that predicts "no churn" for everyone scores 83.9 percent. Notebook 03
should be judged on recall, precision and PR-AUC, with the business trade-off
between them stated explicitly.

**6. `Card_Category` Platinum has 20 rows.** Any importance or coefficient
attached to it is noise and should not be reported as a finding.

**7. `Customer_Age` and `Months_on_book` correlate at 0.79.** Neither predicts
churn (0.02 and 0.01), so this matters little in practice, but they should not
be read as two independent tenure signals.

## 13. Summary and handoff

**Dropped**
- `Attrition_Flag`: string duplicate of the target. Hard leakage.
- `Avg_Open_To_Buy`: exactly `Credit_Limit` minus `Total_Revolving_Bal`. Perfect
  linear collinearity, verified on every row.
- Three "Unknown" dummies: replaced by `has_unknown_demographic`, after testing
  showed no signal is lost.
- `Card_Category_Blue`: reference level, dropped to avoid the dummy variable trap
  in a logistic baseline.

**Engineered**
- `zero_revolving_balance`: 24.4 percent of customers, 36.2 percent churn against
  9.6 percent. Tested and confirmed to be transactors, not dormant accounts.
- `sharp_activity_decline`: Q4 transaction count at or below half of Q1. 15.1
  percent of customers, 49.3 percent churn against 10.1 percent, catching 46.4
  percent of all churners. Fixed rule, not a fitted quantile, so it does not leak
  and it can be deployed on a single new customer.
- `has_unknown_demographic`: tested, found to carry no measurable signal
  (p = 0.15). Retained as one column, with the negative result on the record.

**Rejected, with evidence printed above**
- `avg_transaction_value`: correlates with churn at 0.02 and is non-monotonic
  across quartiles.
- `Total_Trans_Ct` times `Months_Inactive_12_mon`: uninterpretable, inputs move in
  opposite directions.
- Log transforms: tested. `Credit_Limit` gains nothing (-0.024 to -0.045, both
  near zero). `Total_Trans_Amt` does gain (-0.169 to -0.227), but a log is
  invisible to a tree model, so it is applied inside the logistic Pipeline in
  Notebook 03 rather than baked into this shared file.
- Scaling: not applied. A scaler is fitted, so fitting it on the full dataset
  would leak. It belongs in a Pipeline in Notebook 03.

**Output**
- `data/processed/churn_features.csv`: 10,127 rows, 36 columns (1 identifier, 1
  target, 34 predictors).

**Notebook 03 must do the following**
1. Load with `pd.read_csv(OUTPUT_PATH)` and immediately exclude `CLIENTNUM` from
   the feature matrix. It is an account number, not a predictor.
2. Split before doing anything else, stratified on `Churn` (16.07 percent
   positive).
3. Put every fitted transform (scaler, imputer, any target-based encoding) inside a
   `Pipeline` fitted on the training split only. If a logistic baseline is added,
   put `np.log` on `Total_Trans_Amt` in that Pipeline too, via a
   `FunctionTransformer`.
4. Resolve the `zero_revolving_balance` / `Total_Revolving_Bal` /
   `Avg_Utilization_Ratio` overlap there, using feature importance and a
   correlation or VIF check, not by guessing here.
5. Report recall, precision and PR-AUC. Not accuracy.